[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/02_graph_tuning.ipynb)

# 02 — Graph Tuning
Tune `sigma2` and `theta` for the adjacency matrix on the 100-station subset.

In [ ]:
# Clone repo to access preprocessing/graph.py
!git clone https://github.com/skyexry/urban-mobility-forecast.git 2>/dev/null || git -C urban-mobility-forecast pull

In [ ]:
import importlib.util
import sys
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from google.colab import drive

drive.mount('/content/drive')
sys.path.append('/content/urban-mobility-forecast')

Mounted at /content/drive


In [ ]:
spec = importlib.util.spec_from_file_location(
    "graph", "/content/urban-mobility-forecast/preprocessing/graph.py"
)
graph_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(graph_mod)

## 1. Load filtered data (100 stations)

In [ ]:
df = pd.read_parquet('/content/drive/MyDrive/citibike/hourly_demand_filtered.parquet')
df['hour'] = pd.to_datetime(df['hour'])
print(f'Stations : {df["start_station_id"].nunique()}')
print(f'Rows     : {len(df):,}')

Stations : 100
Rows     : 1,435,803


## 2. Distance stats

Distances are computed in **degrees** (raw lat/lng Euclidean), not km.  
At NYC's latitude, 1 degree ≈ 85 km, so the values below translate roughly to:

| Stat | Degrees | ~km |
|------|---------|-----|
| min  | 0.0007  | 60 m |
| mean | 0.0277  | 2.4 km |
| max  | 0.0854  | 7.3 km |

This matters for `sigma2`: since distances are in degrees (small numbers), `sigma2` must also be small (e.g. 0.001) to produce meaningful Gaussian weights. If we used km, `sigma2` would need to be thousands of times larger.

In [ ]:
stations_df = (
    df.groupby('start_station_id')[['start_lat', 'start_lng']]
    .mean().reset_index()
)
coords = stations_df[['start_lat', 'start_lng']].values
dist_matrix = cdist(coords, coords, metric='euclidean')
distances = dist_matrix[dist_matrix > 0]  # exclude self-distances (zeros on diagonal)

print(f'Distance stats (degrees):')
print(f'  min  : {distances.min():.4f} deg  (~{distances.min()*85:.3f} km)')
print(f'  mean : {distances.mean():.4f} deg  (~{distances.mean()*85:.2f} km)')
print(f'  max  : {distances.max():.4f} deg  (~{distances.max()*85:.2f} km)')

Distance stats (degrees):
  min  : 0.0007 deg  (~0.063 km)
  mean : 0.0277 deg  (~2.35 km)
  max  : 0.0854 deg  (~7.26 km)


## 3. Grid search over sigma2 and theta

### Parameters

**`sigma2`** — Gaussian kernel variance. Controls how fast edge weight decays with distance:
- `w_ij = exp(-d² / sigma2)`
- Smaller sigma2 → weights decay faster → only very close stations get high weights
- Must match the distance scale (here: degrees, so sigma2 ~ 0.001)

**`theta`** — Sparsity threshold. Edges with weight below theta are set to 0:
- Higher theta → stricter → fewer edges → sparser graph
- Lower theta → looser → more edges → denser graph

### Decision criteria

Target **avg_degree of 5–15** (out of 100 nodes):
- Too high (>20): graph too dense, spatial locality lost, every node connects to almost everyone
- Too low (<3): graph too sparse, poor information propagation, near-isolated nodes

In [12]:
results = []

# Extended ranges: smaller sigma2 and higher theta to reduce density
for sigma2 in [0.0001, 0.0005, 0.001]:
    for theta in [0.7, 0.9, 0.95, 0.99]:
        W, _ = graph_mod.build_adjacency_matrix(df, sigma2=sigma2, theta=theta)
        n = len(_)
        edges = int(np.count_nonzero(W))
        sparsity = 1 - edges / (n ** 2)
        avg_degree = edges / n
        results.append(dict(sigma2=sigma2, theta=theta, edges=edges,
                            sparsity=sparsity, avg_degree=avg_degree))
        print(f"sigma2={sigma2}, theta={theta}: {edges} edges, "
              f"sparsity={sparsity:.2%}, avg_degree={avg_degree:.1f}")

Stations     : 100
Non-zero edges: 444
Sparsity     : 95.56%
sigma2=0.0001, theta=0.7: 444 edges, sparsity=95.56%, avg_degree=4.4
Stations     : 100
Non-zero edges: 136
Sparsity     : 98.64%
sigma2=0.0001, theta=0.9: 136 edges, sparsity=98.64%, avg_degree=1.4
Stations     : 100
Non-zero edges: 34
Sparsity     : 99.66%
sigma2=0.0001, theta=0.95: 34 edges, sparsity=99.66%, avg_degree=0.3
Stations     : 100
Non-zero edges: 4
Sparsity     : 99.96%
sigma2=0.0001, theta=0.99: 4 edges, sparsity=99.96%, avg_degree=0.0
Stations     : 100
Non-zero edges: 1900
Sparsity     : 81.00%
sigma2=0.0005, theta=0.7: 1900 edges, sparsity=81.00%, avg_degree=19.0
Stations     : 100
Non-zero edges: 656
Sparsity     : 93.44%
sigma2=0.0005, theta=0.9: 656 edges, sparsity=93.44%, avg_degree=6.6
Stations     : 100
Non-zero edges: 314
Sparsity     : 96.86%
sigma2=0.0005, theta=0.95: 314 edges, sparsity=96.86%, avg_degree=3.1
Stations     : 100
Non-zero edges: 34
Sparsity     : 99.66%
sigma2=0.0005, theta=0.99: 34 

## 4. Pick best parameters

The grid search shows avg_degree of 32–98 — far too dense for 100 nodes (ideally 5–15 neighbors per node).  
This happens because the 100 stations are tightly packed in Manhattan: even with `sigma2=0.001`, most pairs are close enough to exceed `theta`.

**Options to reduce density:**
- Raise `theta` above 0.7 (e.g. 0.9, 0.95)
- Lower `sigma2` below 0.001 (e.g. 0.0005)
- Or switch to **k-nearest neighbors** in `graph.py` for direct degree control

In [14]:
results_df = pd.DataFrame(results)
results_df

,sigma2,theta,edges,sparsity,avg_degree
0,0.0001,0.70,444,0.9556,4.44
1,0.0001,0.90,136,0.9864,1.36
2,0.0001,0.95,34,0.9966,0.34
3,0.0001,0.99,4,0.9996,0.04
4,0.0005,0.70,1900,0.8100,19.00
5,0.0005,0.90,656,0.9344,6.56
6,0.0005,0.95,314,0.9686,3.14
7,0.0005,0.99,34,0.9966,0.34
8,0.0010,0.70,3214,0.6786,32.14
9,0.0010,0.90,1194,0.8806,11.94


## 5. Build final graph with chosen parameters

In [15]:
# Set based on grid search results above
SIGMA2 = 0.0005
THETA  = 0.90

W, stations_out = graph_mod.build_adjacency_matrix(df, sigma2=SIGMA2, theta=THETA)
edge_index, edge_weight = graph_mod.adjacency_to_edge_index(W)

print(f'edge_index shape : {edge_index.shape}')
print(f'edge_weight shape: {edge_weight.shape}')

Stations     : 100
Non-zero edges: 656
Sparsity     : 93.44%
edge_index shape : (2, 656)
edge_weight shape: (656,)


## 6. Save

In [16]:
np.save('/content/drive/MyDrive/citibike/W.npy', W)
np.save('/content/drive/MyDrive/citibike/edge_index.npy', edge_index)
np.save('/content/drive/MyDrive/citibike/edge_weight.npy', edge_weight)
stations_out.to_parquet('/content/drive/MyDrive/citibike/stations_graph.parquet', index=False)
print('Saved W.npy, edge_index.npy, edge_weight.npy, stations_graph.parquet')

Saved W.npy, edge_index.npy, edge_weight.npy, stations_graph.parquet
